# Exercice : Analyseur de Logs Multi-niveaux (Advanced)

**Nouveau Contexte :** Les données sont maintenant "sales". Elles contiennent des doublons, des espaces irréguliers, et vous devez effectuer une analyse temporelle simplifiée.

```py

raw_logs = [
    "2026-04-10 10:00:01 | ERROR | Database connection failed",
    "2026-04-10 10:05:24 | INFO | User logged in",
    "2026-04-10 10:05:24 | INFO | User logged in",
    " 2026-04-10 10:10:45 | ERROR | Out of memory  ",
    "2026-04-10 11:15:12 | WARNING | High CPU usage",
    "2026-04-10 11:20:00 | ERROR | Connection timeout",
    "invalid log line" 
]
```
**Nouvelles Instructions de complexité :**

1. Prétraitement & Unicité :

- Supprimez les doublons exacts de la liste (Indice : utilisez set() puis repassez en list).
- Supprimez les espaces superflus au début et à la fin de chaque ligne.

In [15]:
raw_logs = [
    "2026-04-10 10:00:01 | ERROR | Database connection failed",
    "2026-04-10 10:05:24 | INFO | User logged in",
    "2026-04-10 10:05:24 | INFO | User logged in",
    " 2026-04-10 10:10:45 | ERROR | Out of memory  ",
    "2026-04-10 11:15:12 | WARNING | High CPU usage",
    "2026-04-10 11:20:00 | ERROR | Connection timeout",
    "invalid log line" 
]

# 1. Delete duplicated logs
def remove_duplicates(logs):
    return list(set(logs))

# 2. Delete whitespace of each log
def clean_logs(logs):
    return [log.strip() for log in logs]


2. Parsing Robuste :

- Lors du parcours, ignorez les lignes qui ne contiennent pas le séparateur |.
- Utilisez le tuple unpacking pour extraire date_time, level et message.

In [16]:
# 3. Ignore invalid logs (those that don't have the expected format)
def filter_valid_logs(logs):
    valid_logs = []
    invalid_count = 0
    for log in logs:
        if '|' not in log:
            invalid_count += 1
            continue
        valid_logs.append(log)

    return valid_logs, invalid_count

# Parse logs into structured data

def parse_logs(log):
    date_time, level, message = [part.strip() for part in log.split("|")]
    return date_time, level, message

3. Analyse Temporelle :

- Extrayez uniquement l'heure (les deux premiers chiffres après la date) pour chaque log.
- Créez un dictionnaire logs_par_heure qui compte combien de logs ont eu lieu à chaque heure (ex: {"10h": 3, "11h": 2}).

In [17]:
# Count logs by hour
def count_logs_by_hour(logs):
    logs_per_hour = {}
    for log in logs:
        # Extract hour from datetime
        date_time, level, message = parse_logs(log)
        hour = date_time.split(' ')[1][:2] + 'h' # Get hour part and append 'h'
        logs_per_hour[hour] = logs_per_hour.get(hour, 0) + 1
    return logs_per_hour


4. Filtrage Avancé :

- Générez un dictionnaire nommé critical_reports uniquement pour le niveau ERROR. Ce dictionnaire doit avoir pour clé le timestamp et pour valeur le message.

In [18]:
# Extract critical reports (ERROR level)
def extract_critical_reports(logs):
    critical_reports = {}
    for log in logs:
        date_time, level, message = parse_logs(log)
        if level == "ERROR":
            critical_reports[date_time] = message
    return critical_reports


5. Affichage Final :

- Affichez le nombre total de lignes ignorées (doublons + erreurs de format).
- Affichez la liste des messages uniques (sans doublons) de niveau ERROR.

In [20]:
def get_unique_error_messages(critical_reports):
    return list(set(critical_reports.values()))


unique_logs = remove_duplicates(raw_logs)
duplicate_count = len(raw_logs) - len(unique_logs)

cleaned_logs = clean_logs(unique_logs)
valid_logs, invalid_count = filter_valid_logs(cleaned_logs)

logs_per_hour = count_logs_by_hour(valid_logs)
critical_reports = extract_critical_reports(valid_logs)
error_messages = get_unique_error_messages(critical_reports)

ignored_count = duplicate_count + invalid_count

print('Logs par heure :', logs_per_hour)
print('Critical Reports :', critical_reports)
print('Nombre de lignes ignorees :', ignored_count)
print('Messages ERROR uniques :', error_messages)

Logs par heure : {'11h': 2, '10h': 3}
Critical Reports : {'2026-04-10 11:20:00': 'Connection timeout', '2026-04-10 10:00:01': 'Database connection failed', '2026-04-10 10:10:45': 'Out of memory'}
Nombre de lignes ignorees : 2
Messages ERROR uniques : ['Connection timeout', 'Out of memory', 'Database connection failed']
